### Multi-Document RAG-Based Question Answering System using LangChain, FAISS & Ollama

In [3]:

from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
import faiss

import warnings
warnings.filterwarnings("ignore")

In [4]:


DOCS_PATH = "docs2"

loader = DirectoryLoader(
    path=DOCS_PATH,
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
)

documents = loader.load()

print(f" Loaded {len(documents)} pages from all PDFs")


 Loaded 5 pages from all PDFs


In [5]:

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

split_docs = text_splitter.split_documents(documents)
print(f" Split into {len(split_docs)} chunks")


 Split into 25 chunks


In [6]:

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

embedding_dim = 384
index = faiss.IndexFlatIP(embedding_dim)

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

vector_store.add_documents(split_docs)
print(" Documents embedded and indexed in FAISS")


retriever = vector_store.as_retriever(
    search_kwargs={"k": 5}
)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4477.47it/s]


 Documents embedded and indexed in FAISS


In [7]:


llm = ChatOllama(model="llama3.2")


In [8]:

query = "What is the USA GDP in 2025"

retrieved_docs = retriever.invoke(query)

context_text = "\n\n".join(
    [doc.page_content for doc in retrieved_docs]
)


In [9]:

prompt = ChatPromptTemplate.from_template("""
You are an expert assistant.

INSTRUCTIONS:
- Answer ONLY from the provided context.
- Do NOT use outside knowledge.
- If the answer is not found, say:
  "The information is not available in the provided documents."

CONTEXT:
{context}

QUESTION:
{question}

Provide a structured bullet-point answer.
""")

formatted_prompt = prompt.format(
    context=context_text,
    question=query
)


In [10]:

response = llm.invoke(formatted_prompt)

print("\n Answer:\n")

print(response.content)



 Answer:

Here is the answer to the question in a structured bullet-point format:

• The estimated USA GDP in 2025 is in the range of $29–30 trillion USD.
• The moderate pace of growth is approximately 1.8% to 2.3% annually.


In [11]:

print("\n Retrieved Sources:\n")

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"--- Chunk {i} ---")
    print(f"Source File: {doc.metadata.get('source', 'N/A')}")
    print(f"Page Number: {doc.metadata.get('page', 'N/A')}")
    print(doc.page_content[:400], "...\n")



 Retrieved Sources:

--- Chunk 1 ---
Source File: docs2\USA2.pdf
Page Number: 0
U.S. GDP in 2025 — Economic Overview 
Nominal GDP Estimate (2025) 
 
As of 2025, the United States’ nominal Gross Domestic Product (GDP) is estimated to 
be in the range of $29–30 trillion USD, maintaining its position as the largest economy in 
the world. This growth is driven by strong consumer spending, continued expansion in 
the services sector, and large-scale investments in technology, ener ...

--- Chunk 2 ---
Source File: docs2\USA2.pdf
Page Number: 0
manufacturing. 
 
Economic Growth Trends 
 
In 2025, the U.S. economy is projected to grow at a moderate pace of approximately 
1.8% to 2.3% annually. This growth reflects a balance between cooling inflation, 
stabilized interest rates, and sustained demand in key industries such as artificial 
intelligence, healthcare, defense, and clean energy. 
 
Sector Contributions to GDP 
 
The U.S. GDP rema ...

--- Chunk 3 ---
Source File: docs2\USA2.pdf
Page